# Baseline 全自动批跑（Colab）

在 Google Colab 挂载本地同步的 `MyDrive/education_video`，**运行时 → 全部运行** 即可。

依次执行：manim → remotion → c2v → tm → eval。已完成的 topic 自动跳过。

- 统一使用 **OpenAI API Key**（`OPENAI_API_KEY`）
- 全程 **实时打印 API 费用估算**（`[usage] est. $...`）
- Code2Video LLM 并发：**20**

**准备**：`topics.xlsx`、`eval.ipynb`、`baseline/runs/api_usage_tracker.py`、`baseline/runs/colab_bootstrap.py`；tm 需 GPU + `TeachingMonster-released`。

In [ ]:
# 1) Mount Drive + OpenAI key + API usage tracking

import os
import sys
from pathlib import Path

from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/education_video")
RUNS_DIR = DRIVE_ROOT / "baseline" / "runs"
EVAL_NB = DRIVE_ROOT / "eval.ipynb"

if not RUNS_DIR.exists():
    raise FileNotFoundError(f"Missing baseline/runs: {RUNS_DIR}")
if not EVAL_NB.exists():
    raise FileNotFoundError(f"Missing eval.ipynb: {EVAL_NB}")
if not any((folder / "api_usage_tracker.py").exists() for folder in (RUNS_DIR, DRIVE_ROOT)):
    raise FileNotFoundError(f"Missing api_usage_tracker.py in {RUNS_DIR} or {DRIVE_ROOT}")

sys.path.insert(0, str(RUNS_DIR))
os.environ["BASELINE_MAX_CONCURRENT_LLM"] = "20"
os.environ["BASELINE_USAGE_FILE"] = str(DRIVE_ROOT / "baseline" / "api_usage_session.json")
os.environ.setdefault("OPENAI_MODEL", "gpt-4o")

from colab_bootstrap import ensure_openai_key, setup_api_tracking, print_api_usage

ensure_openai_key()
setup_api_tracking()
print_api_usage("session start")
print("Drive root:", DRIVE_ROOT)
print("OPENAI_MODEL:", os.environ["OPENAI_MODEL"])

In [ ]:
# 2) Install notebook executor

import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nbformat", "nbclient", "openai", "openpyxl"])

In [ ]:
# 3) Execute baseline notebooks sequentially

import traceback
from datetime import datetime

import nbformat
from nbclient import NotebookClient
from colab_bootstrap import print_api_usage

PIPELINES = [
    "manim.ipynb",
    "remotion.ipynb",
    "c2v_colab.ipynb",
    "tm_colab.ipynb",
]

LOG_DIR = DRIVE_ROOT / "baseline" / "auto_run_logs" / datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR.mkdir(parents=True, exist_ok=True)
exit_codes = {}

for name in PIPELINES:
    nb_path = RUNS_DIR / name
    if not nb_path.exists():
        print(f"SKIP missing notebook: {nb_path}")
        exit_codes[name] = "missing"
        continue

    print("\n" + "=" * 80)
    print(f"START {name}")
    print("=" * 80)

    try:
        nb = nbformat.read(nb_path, as_version=4)
        client = NotebookClient(nb, timeout=None, kernel_name="python3")
        client.execute()
        nbformat.write(nb, LOG_DIR / f"executed_{name}")
        exit_codes[name] = 0
        print(f"END {name} ok")
    except Exception as exc:
        exit_codes[name] = 1
        (LOG_DIR / f"error_{name}.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print(f"END {name} FAILED: {exc}")

    print_api_usage(name)

print("\nPipeline exit codes:", exit_codes)
print_api_usage("after baselines")

In [ ]:
# 4) Eval + summary table

import traceback

import nbformat
from nbclient import NotebookClient
from colab_bootstrap import print_api_usage

print("\n" + "=" * 80)
print("START eval.ipynb")
print("=" * 80)

try:
    nb = nbformat.read(EVAL_NB, as_version=4)
    client = NotebookClient(nb, timeout=None, kernel_name="python3")
    client.execute()
    nbformat.write(nb, LOG_DIR / "executed_eval.ipynb")
    print("END eval.ipynb ok")
    print("Pivot table:", DRIVE_ROOT / "eval_outputs" / "latest_baseline_metric_average_pivot.xlsx")
except Exception as exc:
    (LOG_DIR / "error_eval.txt").write_text(traceback.format_exc(), encoding="utf-8")
    print("END eval.ipynb FAILED:", exc)

print_api_usage("final")